# GISLR — Stage 1 — phonology-only streaming models, trained in parallel (TODO §3.10)

**Pipeline stage (training), run by the user.** The user's request (2026-09-26): train only on the
phonological features, only models that stream, as fast as the machine allows.

- **Input:** the 130 per-frame phonological features of `sb.recognize.phonology` (feature pipeline
  `phono130_v1`, all 543 landmarks in, no raw coordinates). Cache built 2026-09-26 (key `157b4b1550f45acf`).
- **Models (all streaming):** `gru_phono130` (`StreamingGRU`), `lstm_phono130` (`StreamingLSTM`),
  `cnn1d_phono130` (causal dilated CNN). `bilstm_phono130` is parked (`enabled: false`, offline-only).
- **Speed:** the three train **at the same time**, one process each on the one GPU. They are small and the
  cache sits in RAM, so a single run leaves the GPU mostly idle. `es_patience` 10 for these arms (config).
- **Hyperparameters:** only in `experiments/recognition/configs/gislr.training.json` (`shared` +
  per-arch `overrides`), same regime as every `gru` run.

| artifact | path |
|---|---|
| run folders | `registry/runs/<run_id>/` (one per arch) |
| training logs | `data/temp/phono130_train_<arch>.log` |
| comparison table | printed in §3; runs scored canonically into their `meta.json` |

**Resumable:** re-running §2 restarts any process that is not finished; the driver resumes each run from
its `last.pt`. §3 only scores finished runs.

Baselines to beat (canonical, current split): `gru_phono_raw` ME_134 **0.7632**, `gru` ME_132 **0.7517**;
no training, the same 130 features give 41.4% (template) / 54.3% (1-NN).

## 1. Setup

In [ ]:
# ============================================================
# Imports, tunables, which runs exist
# ============================================================
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from sb.core.paths import ROOT_DIR, TEMP_DIR
from sb.mlops import registry as R
from sb.recognize.config import load_config
from sb.recognize.data import subset_tag

ARCHS = ["gru_phono130", "lstm_phono130", "cnn1d_phono130"]  # streaming-capable only
POLL_S = 20                    # seconds between progress checks
BASELINES = {"gru_phono_raw ME_134": 0.7632, "gru ME_132": 0.7517}

CFG = load_config()
ARCHS = [a for a in ARCHS if CFG.enabled(a)]
LOGS = {a: TEMP_DIR / f"phono130_train_{a}.log" for a in ARCHS}
TEMP_DIR.mkdir(parents=True, exist_ok=True)


def run_dir(arch):
    """The run folder the registry pointer names for this arch (None before it starts)."""
    key = f"{CFG.dataset}_{arch}_{subset_tag(CFG.subsets_for(arch)[0], CFG.coords_for(arch))}"
    return R.pointer_run_dir(key)


for a in ARCHS:
    print(f"{a:16s} overrides {CFG.overrides_for(a)}  run {run_dir(a)}")

## 2. Train all three at once

One `train_from_config` process per architecture, started together, output to its log. The bar tracks the
epochs finished across all runs (from each run's `meta.json`); a run ends at its early stop.

In [ ]:
# ============================================================
# Parallel training: one process per architecture on the one GPU
# ============================================================
procs = {}
for a in ARCHS:
    d = run_dir(a)
    if d is not None and R.load_meta(d)["training"].get("finished"):
        print(f"{a}: already finished ({d.name})")
        continue
    cmd = [sys.executable, "-c", f"from sb.recognize.train import train_from_config; train_from_config({a!r})"]
    procs[a] = subprocess.Popen(cmd, cwd=ROOT_DIR, stdout=open(LOGS[a], "w"), stderr=subprocess.STDOUT)
    print(f"{a}: started (pid {procs[a].pid}), log {LOGS[a]}")

bar = tqdm(desc="epochs (all runs)", unit="ep")
while procs and any(p.poll() is None for p in procs.values()):
    time.sleep(POLL_S)
    state = {}
    for a in ARCHS:
        d = run_dir(a)
        if d is not None and (d / "meta.json").exists():
            m = R.load_meta(d)
            state[a] = (m["training"]["epochs_trained"], m["metrics"]["train_val_acc"])
    bar.n = sum(e for e, _ in state.values())
    bar.set_postfix_str(" | ".join(f"{a.split('_')[0]} ep{e} {v:.3f}" for a, (e, v) in state.items()))
    bar.refresh()
bar.close()
for a, p in procs.items():
    print(f"{a}: exit code {p.returncode}" + ("" if p.returncode == 0 else f" -> see {LOGS[a]}"))

## 3. Canonical evaluation and comparison

`sb-evaluate` on every finished run (the canonical 18,896-clip split, per-class accuracy; the checkpoint
says `features: phono130_v1`, so the evaluator rebuilds the same features), then the table against the
baselines.

In [ ]:
# ============================================================
# Score finished runs canonically; compare
# ============================================================
from sb.recognize.evaluate import evaluate_run

rows = []
for a in ARCHS:
    d = run_dir(a)
    if d is None:
        continue
    m = R.load_meta(d)
    if not m["training"].get("finished"):
        print(f"{a}: not finished ({m['training']['epochs_trained']} epochs) — run §2 again")
        continue
    if m["metrics"].get("eval_status") != "canonical":
        evaluate_run(d, verbose=False)
        m = R.load_meta(d)
    rows.append({"arch": a, "run": d.name, "params": m["n_params"], "epochs": m["training"]["epochs_trained"],
                 "canonical_acc": m["metrics"]["overall_accuracy"], "macro_acc": m["metrics"]["macro_accuracy"],
                 "classes_below_50pct": m["metrics"]["n_classes_below_50pct"],
                 "wall_min": m["training"]["wall_time_min"]})
table = pd.DataFrame(rows)
for name, acc in BASELINES.items():
    table[f"vs {name}"] = table.canonical_acc - acc
print(table.round(4).to_string(index=False))
print("\nthen: .venv/Scripts/sb-docs.exe (refresh index/README) and push the checkpoints: sb-sync push --apply")